In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
data = pd.read_csv('train.csv')

In [3]:
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
data = np.array(data)
#when training a ml model, you can easily run into issues where the model becomes good at predicting based off your model data but not the general case
# therefore, you split up your data and don't train on all of it so you can use part of the data as a "cross validation" portion of your data (so you can introduce novel cases to the model).
m, n = data.shape #m = num rows, n = features + 1
np.random.shuffle(data)

data_dev = data[0:1000].T
Y_dev = data_dev[0]
X_dev = data_dev[1:n] / 255.

data_train = data[1000:m].T
Y_train = data_train[0]
X_train = data_train[1:n] / 255.


## Forward prop model

input layer: $x = A^{[0]}$

hidden layer: $z^{[1]} = w^{[1]}A^{[0]} + b^{[1]}$

hidden layer input: $A^{[1]} = g(z^{[1]}) = ReLu(z^{[1]})$
$\ ReLu(x) = x $ if $x > 0$ 

(and $ReLu(x) = 0$ if $x \leq 0$)

unactivated output layer: $z^{[2]} = w^{[2]}A^{[1]} + b^{[2]}$

Softmax step to transform the output layer ($z^{[2]}) to a vector of probabilities (the highest one will be our prediction$

$A^{[2]} = \sigma(z^{[2]})$ where

$\sigma(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{k}e^{z_j}}$

## Back prop model

The backward propogation model dictates how we generate  biases. We will calculate the gradients for each w, z and b then perform gradient descent with a hard-coded learning rate $\alpha$

$dz^{[2]} = A^{[2]} - Y$

where Y is the one-hot encoded prediction so for $y=4, \  Y = <0, 0, 0, 0, 1, ... ,0>$ where the 1 is in the index $j = 4$

$dw^{[2]} = \frac{1}{m} dz^{[2]} {A^{[1]}}^T$

$db^{[2]} = \frac{1}{m} \sum dz^{[2]}$

$$

$$

$dw^{[1]} = $

$db^{[1]} = \frac{1}{m} \sum dz^{[1]}$



In [5]:
#initialize parameters
def init_params():
    W1 = np.random.rand(10, 784) - 0.5
    b1 = np.random.rand(10, 1) - 0.5
    W2 = np.random.rand(10, 10) - 0.5
    b2 = np.random.rand(10, 1) - 0.5
    return W1, b1, W2, b2

def ReLu(Z):
    return np.maximum(0, Z)

def softmax(Z):
    Z = Z - np.max(Z, axis=0, keepdims=True)
    expZ = np.exp(Z)
    return expZ / np.sum(expZ, axis=0, keepdims=True)
    
def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 = ReLu(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y
    return one_hot_Y
def deriv_ReLU(Z):
    return Z > 0
    
def back_prop(Z1, A1, Z2, A2, W2, X, Y):
    m = Y.size
    one_hot_Y = one_hot(Y)
    dZ2 = A2 -  one_hot_Y
    dW2 = 1 / m * dZ2.dot(A1.T)
    db2 = 1 / m * np.sum(dZ2, axis=1, keepdims=True)
    dZ1 = W2.T.dot(dZ2) * deriv_ReLU(Z1)

    dW1 = 1 / m * dZ1.dot(X.T)
    db1 = 1 / m * np.sum(dZ1, axis=1, keepdims=True)
    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2
    return W1, b1, W2, b2

In [6]:
def get_predictions(A2):
    return np.argmax(A2, 0) #return the largest value column wise from the second hidden layer

def get_accuracy(predictions, Y):
    print(predictions, Y)
    return np.sum(predictions == Y) / Y.size
def gradient_descent(X, Y, iterations, alpha):
    W1, b1, W2, b2 = init_params()
    for i in range(iterations):
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2, db2 = back_prop(Z1, A1, Z2, A2, W2, X, Y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
        if i % 100 == 0:
            print("iteration: ", i)
            print("Accuracy: ", get_accuracy(get_predictions(A2), Y))
    return W1, b1, W2, b2

In [ ]:
W1, b1, W2, b2 = gradient_descent(X_train, Y_train, 60000, 0.01) 

iteration:  0
[8 4 4 ... 3 4 8] [5 6 1 ... 4 7 1]
Accuracy:  0.10065853658536586
iteration:  100
[8 8 8 ... 3 1 8] [5 6 1 ... 4 7 1]
Accuracy:  0.1846341463414634
iteration:  200
[8 8 8 ... 3 1 8] [5 6 1 ... 4 7 1]
Accuracy:  0.24239024390243902
iteration:  300
[8 8 8 ... 3 8 8] [5 6 1 ... 4 7 1]
Accuracy:  0.2857073170731707
iteration:  400
[8 6 8 ... 3 8 8] [5 6 1 ... 4 7 1]
Accuracy:  0.3026829268292683
iteration:  500
[8 6 8 ... 3 8 8] [5 6 1 ... 4 7 1]
Accuracy:  0.3241951219512195
iteration:  600
[8 6 8 ... 7 8 8] [5 6 1 ... 4 7 1]
Accuracy:  0.3648048780487805
iteration:  700
[8 6 8 ... 9 8 8] [5 6 1 ... 4 7 1]
Accuracy:  0.419
iteration:  800
[8 6 8 ... 9 8 8] [5 6 1 ... 4 7 1]
Accuracy:  0.4774390243902439
iteration:  900
[8 6 1 ... 9 2 8] [5 6 1 ... 4 7 1]
Accuracy:  0.5277804878048781
iteration:  1000
[3 6 1 ... 9 2 8] [5 6 1 ... 4 7 1]
Accuracy:  0.570170731707317
iteration:  1100
[3 6 1 ... 4 2 8] [5 6 1 ... 4 7 1]
Accuracy:  0.6048780487804878
iteration:  1200
[3 6 1 ... 